In [2]:
# ============================================
# UFC DATA CLEANING AND PROCESSING
# Input: ufc_complete_data-4.csv
# Output: ufc_stance_handedness_analysis.xlsx (4 sheets)
# ============================================

import pandas as pd
import numpy as np
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("UFC DATA CLEANING AND PROCESSING")
print("CSV to Analysis Excel (4 sheets)")
print("="*60)

# ============================================
# STEP 1: LOAD CSV FILE
# ============================================
print("\nSTEP 1: Loading CSV file...")

df_raw = pd.read_csv('ufc_complete_data-4.csv')
print(f"Loaded {len(df_raw)} fighters")

print(f"   Columns: {df_raw.columns.tolist()}")

# ============================================
# STEP 2: CLEAN DATA
# ============================================
print("\nSTEP 2: Cleaning data...")

# Standardize column names
df_raw.columns = df_raw.columns.str.strip()

# Fill missing values
df_raw['stance'] = df_raw['stance'].fillna('unknown')
df_raw['handedness'] = df_raw['handedness'].fillna('unknown')
df_raw['height_cm'] = df_raw['height_cm'].fillna(df_raw['height_cm'].median())
df_raw['reach_cm'] = df_raw['reach_cm'].fillna(df_raw['reach_cm'].median())
df_raw['weight_kg'] = df_raw['weight_kg'].fillna(df_raw['weight_kg'].median())
df_raw['win_rate'] = df_raw['win_rate'].fillna(0.5)

# Ensure win_rate_group exists
if 'win_rate_group' not in df_raw.columns:
    def get_win_group(rate):
        if rate >= 0.9: return '90%+'
        elif rate >= 0.8: return '80-90%'
        elif rate >= 0.7: return '70-80%'
        elif rate >= 0.6: return '60-70%'
        elif rate >= 0.5: return '50-60%'
        elif rate >= 0.4: return '40-50%'
        else: return '<40%'
    df_raw['win_rate_group'] = df_raw['win_rate'].apply(get_win_group)

# Create total_fights if missing
if 'total_fights' not in df_raw.columns:
    df_raw['total_fights'] = df_raw['wins'] + df_raw['losses'] + df_raw['draws']

# Remove rows with empty names
df_raw = df_raw[df_raw['fighter_name'].notna()]
df_raw = df_raw[df_raw['fighter_name'] != '']

print(f"After cleaning: {len(df_raw)} fighters")

# ============================================
# STEP 3: CREATE STATISTICS GROUPS
# ============================================
print("\nSTEP 3: Creating statistics groups...")

# Filter orthodox/southpaw with right/left
filt = df_raw[
    (df_raw['stance'].isin(['orthodox', 'southpaw'])) &
    (df_raw['handedness'].isin(['right', 'left']))
].copy()

g_ort_right = filt[(filt['stance'] == 'orthodox') & (filt['handedness'] == 'right')]
g_ort_left = filt[(filt['stance'] == 'orthodox') & (filt['handedness'] == 'left')]
g_sp_left = filt[(filt['stance'] == 'southpaw') & (filt['handedness'] == 'left')]
g_sp_right = filt[(filt['stance'] == 'southpaw') & (filt['handedness'] == 'right')]
g_orthodox = filt[filt['stance'] == 'orthodox']
g_southpaw = filt[filt['stance'] == 'southpaw']

def safe_stats(g):
    n = len(g)
    if n == 0:
        return {'n': 0, 'pct': 0, 'mean': 0, 'median': 0, 'std': 0}
    total = len(filt) if len(filt) > 0 else 1
    return {
        'n': n,
        'pct': n / total,
        'mean': g['win_rate'].mean() * 100,
        'median': g['win_rate'].median() * 100,
        'std': g['win_rate'].std() * 100 if n > 1 else 0
    }

# Southpaw Right-Handed fighters (your profile)
sp_right = df_raw[(df_raw['stance'] == 'southpaw') & (df_raw['handedness'] == 'right')].copy()
sp_right = sp_right.sort_values('win_rate', ascending=False).reset_index(drop=True)

print(f"Orthodox + Right: {len(g_ort_right)}")
print(f"Southpaw + Right (your profile): {len(g_sp_right)}")
print(f"Southpaw + Left: {len(g_sp_left)}")

# ============================================
# STEP 4: CREATE EXCEL FILE WITH 4 SHEETS
# ============================================
print("\nSTEP 4: Creating Excel file...")

OUTPUT_FILE = 'ufc_stance_handedness_analysis.xlsx'

# Tableau color palette
BG_TITLE = '1A3A5C'
BG_HEADER = '2C5F8A'
BG_ROW1 = 'F5F5F5'
BG_ROW2 = 'E8EBED'
BG_STAR = 'FFE4B5'
C_WHITE = 'FFFFFF'

wb = Workbook()
wb.remove(wb.active)

# ========== SHEET 1: Fighters Database ==========
ws1 = wb.create_sheet("Fighters Database")
ws1.sheet_view.showGridLines = False

ws1.merge_cells("A1:J1")
c = ws1.cell(1, 1)
c.value = "UFC STANCE INTELLIGENCE REPORT - RIGHT-HANDED SOUTHPAW EDITION"
c.font = Font(size=14, bold=True, color=C_WHITE)
c.fill = PatternFill("solid", fgColor=BG_TITLE)
c.alignment = Alignment(horizontal="center")
ws1.row_dimensions[1].height = 35

headers = ['#', 'Fighter Name', 'Stance', 'Hand', 'Wins', 'Losses', 'Total Fights', 'Win Rate %', 'Win Rate Group', 'Height cm']
for col, h in enumerate(headers, 1):
    cell = ws1.cell(2, col)
    cell.value = h
    cell.font = Font(bold=True, color=C_WHITE)
    cell.fill = PatternFill("solid", fgColor=BG_HEADER)
    cell.alignment = Alignment(horizontal="center")
ws1.row_dimensions[2].height = 25

df_sorted = df_raw.sort_values('win_rate', ascending=False).reset_index(drop=True)

for i, row in df_sorted.iterrows():
    r = i + 3
    is_even = i % 2 == 0
    is_star = row['stance'] == 'southpaw' and row['handedness'] == 'right'
    bg = BG_STAR if is_star else (BG_ROW1 if is_even else BG_ROW2)

    ws1.cell(r, 1, i+1).fill = PatternFill("solid", fgColor=bg)
    ws1.cell(r, 1).alignment = Alignment(horizontal="center")

    ws1.cell(r, 2, row['fighter_name']).fill = PatternFill("solid", fgColor=bg)
    ws1.cell(r, 3, row['stance'].capitalize()).fill = PatternFill("solid", fgColor=bg)
    ws1.cell(r, 4, row['handedness'].capitalize()).fill = PatternFill("solid", fgColor=bg)
    ws1.cell(r, 5, row['wins']).fill = PatternFill("solid", fgColor=bg)
    ws1.cell(r, 6, row['losses']).fill = PatternFill("solid", fgColor=bg)

    ws1.cell(r, 7, f"=E{r}+F{r}").fill = PatternFill("solid", fgColor=bg)
    ws1.cell(r, 7).alignment = Alignment(horizontal="center")

    ws1.cell(r, 8, f"=IF(G{r}>0,E{r}/G{r}*100,0)").fill = PatternFill("solid", fgColor=bg)
    ws1.cell(r, 8).alignment = Alignment(horizontal="center")
    ws1.cell(r, 8).number_format = "0.00"

    ws1.cell(r, 9, row['win_rate_group']).fill = PatternFill("solid", fgColor=bg)
    ws1.cell(r, 10, round(row['height_cm'],1) if pd.notna(row['height_cm']) else 'N/A').fill = PatternFill("solid", fgColor=bg)

    ws1.row_dimensions[r].height = 18

widths = [5, 25, 12, 10, 7, 8, 12, 12, 15, 10]
for i, w in enumerate(widths, 1):
    ws1.column_dimensions[get_column_letter(i)].width = w

# ========== SHEET 2: Summary Stats ==========
ws2 = wb.create_sheet("Summary Stats")
ws2.sheet_view.showGridLines = False

ws2.merge_cells("A1:F1")
ws2.cell(1, 1, "STANCE ANALYSIS - SUMMARY STATISTICS").font = Font(size=14, bold=True, color=C_WHITE)
ws2.cell(1, 1).fill = PatternFill("solid", fgColor=BG_TITLE)
ws2.cell(1, 1).alignment = Alignment(horizontal="center")
ws2.row_dimensions[1].height = 35

ws2.merge_cells("A3:F3")
ws2.cell(3, 1, "GROUP PERFORMANCE COMPARISON").font = Font(bold=True, color=BG_HEADER)
ws2.row_dimensions[3].height = 22

group_headers = ["Group", "N", "% of Dataset", "Mean Win%", "Median Win%", "Std Dev"]
for col, h in enumerate(group_headers, 1):
    cell = ws2.cell(4, col)
    cell.value = h
    cell.font = Font(bold=True, color=C_WHITE)
    cell.fill = PatternFill("solid", fgColor=BG_HEADER)
    cell.alignment = Alignment(horizontal="center")

groups = [
    ("Orthodox + Right-handed", g_ort_right, False),
    ("Orthodox + Left-handed", g_ort_left, False),
    ("Southpaw + Left-handed", g_sp_left, False),
    ("Southpaw + Right-handed", g_sp_right, True)
]

for i, (name, g, is_star) in enumerate(groups, start=5):
    stats = safe_stats(g)
    bg = BG_STAR if is_star else (BG_ROW1 if i % 2 == 0 else BG_ROW2)
    ws2.cell(i, 1, name).fill = PatternFill("solid", fgColor=bg)
    ws2.cell(i, 2, stats['n']).fill = PatternFill("solid", fgColor=bg)
    ws2.cell(i, 3, stats['pct']).fill = PatternFill("solid", fgColor=bg)
    ws2.cell(i, 3).number_format = "0.00%"
    ws2.cell(i, 4, stats['mean']).fill = PatternFill("solid", fgColor=bg)
    ws2.cell(i, 4).number_format = "0.00"
    ws2.cell(i, 5, stats['median']).fill = PatternFill("solid", fgColor=bg)
    ws2.cell(i, 5).number_format = "0.00"
    ws2.cell(i, 6, stats['std']).fill = PatternFill("solid", fgColor=bg)
    ws2.cell(i, 6).number_format = "0.00"
    ws2.row_dimensions[i].height = 18

for i, w in enumerate([30, 10, 15, 12, 12, 12], 1):
    ws2.column_dimensions[get_column_letter(i)].width = w

# ========== SHEET 3: Win Rate Analysis ==========
ws3 = wb.create_sheet("Win Rate Analysis")
ws3.sheet_view.showGridLines = False

ws3.merge_cells("A1:E1")
ws3.cell(1, 1, "WIN RATE GROUP BREAKDOWN").font = Font(size=14, bold=True, color=C_WHITE)
ws3.cell(1, 1).fill = PatternFill("solid", fgColor=BG_TITLE)
ws3.cell(1, 1).alignment = Alignment(horizontal="center")
ws3.row_dimensions[1].height = 35

group_headers = ["Win Rate Group", "Fighters", "Total Wins", "Avg Win %", "% of Total"]
for col, h in enumerate(group_headers, 1):
    cell = ws3.cell(3, col)
    cell.value = h
    cell.font = Font(bold=True, color=C_WHITE)
    cell.fill = PatternFill("solid", fgColor=BG_HEADER)
    cell.alignment = Alignment(horizontal="center")
ws3.row_dimensions[3].height = 22

groups_order = ["90%+", "80-90%", "70-80%", "60-70%", "50-60%", "40-50%", "<40%"]
total_fighters = len(df_raw)

for i, grp in enumerate(groups_order, start=4):
    group_data = df_raw[df_raw['win_rate_group'] == grp]
    n = len(group_data)
    total_wins = group_data['wins'].sum()
    avg_win = group_data['win_rate'].mean() * 100 if n > 0 else 0
    pct = n / total_fighters * 100 if total_fighters > 0 else 0
    bg = BG_ROW1 if i % 2 == 0 else BG_ROW2

    ws3.cell(i, 1, grp).fill = PatternFill("solid", fgColor=bg)
    ws3.cell(i, 2, n).fill = PatternFill("solid", fgColor=bg)
    ws3.cell(i, 3, total_wins).fill = PatternFill("solid", fgColor=bg)
    ws3.cell(i, 4, avg_win).fill = PatternFill("solid", fgColor=bg)
    ws3.cell(i, 4).number_format = "0.00"
    ws3.cell(i, 5, pct).fill = PatternFill("solid", fgColor=bg)
    ws3.cell(i, 5).number_format = "0.00"
    ws3.row_dimensions[i].height = 18

for i, w in enumerate([18, 10, 12, 12, 11], 1):
    ws3.column_dimensions[get_column_letter(i)].width = w

# ========== SHEET 4: Southpaw Right-Handed ==========
ws4 = wb.create_sheet("Southpaw Right-Handed")
ws4.sheet_view.showGridLines = False

ws4.merge_cells("A1:K1")
ws4.cell(1, 1, "SPOTLIGHT: RIGHT-HANDED SOUTHPAW FIGHTERS").font = Font(size=14, bold=True, color=C_WHITE)
ws4.cell(1, 1).fill = PatternFill("solid", fgColor=BG_TITLE)
ws4.cell(1, 1).alignment = Alignment(horizontal="center")
ws4.row_dimensions[1].height = 35

sp_headers = ["#", "Fighter", "Stance", "Hand", "Wins", "Losses", "Total", "Win Rate %", "", "MY GROUP STATS", ""]
for col, h in enumerate(sp_headers, 1):
    cell = ws4.cell(2, col)
    if h != "":
        cell.value = h
        cell.font = Font(bold=True, color=C_WHITE)
        cell.fill = PatternFill("solid", fgColor=BG_HEADER)
        cell.alignment = Alignment(horizontal="center")
ws4.row_dimensions[2].height = 25

for i, row in sp_right.iterrows():
    r = i + 3
    bg = BG_STAR
    ws4.cell(r, 1, i+1).fill = PatternFill("solid", fgColor=bg)
    ws4.cell(r, 1).alignment = Alignment(horizontal="center")
    ws4.cell(r, 2, row['fighter_name']).fill = PatternFill("solid", fgColor=bg)
    ws4.cell(r, 3, row['stance'].capitalize()).fill = PatternFill("solid", fgColor=bg)
    ws4.cell(r, 4, row['handedness'].capitalize()).fill = PatternFill("solid", fgColor=bg)
    ws4.cell(r, 5, row['wins']).fill = PatternFill("solid", fgColor=bg)
    ws4.cell(r, 6, row['losses']).fill = PatternFill("solid", fgColor=bg)
    ws4.cell(r, 7, f"=E{r}+F{r}").fill = PatternFill("solid", fgColor=bg)
    ws4.cell(r, 7).alignment = Alignment(horizontal="center")
    ws4.cell(r, 8, f"=IF(G{r}>0,E{r}/G{r}*100,0)").fill = PatternFill("solid", fgColor=bg)
    ws4.cell(r, 8).alignment = Alignment(horizontal="center")
    ws4.cell(r, 8).number_format = "0.00"
    ws4.row_dimensions[r].height = 18

sp_stats = safe_stats(sp_right)
total_filt = len(filt) if len(filt) > 0 else 1

stats_data = [
    ("Fighters", len(sp_right)),
    ("% of Dataset", f"{len(sp_right)/total_filt*100:.1f}%"),
    ("Mean Win Rate", f"{sp_stats['mean']:.2f}%"),
    ("Median Win Rate", f"{sp_stats['median']:.2f}%"),
    ("Std Deviation", f"{sp_stats['std']:.2f}%"),
    ("Min Win Rate", f"{sp_right['win_rate'].min()*100:.2f}%" if len(sp_right) > 0 else "N/A"),
    ("Max Win Rate", f"{sp_right['win_rate'].max()*100:.2f}%" if len(sp_right) > 0 else "N/A"),
    (None, None),
    ("vs Orthodox Right", f"{sp_stats['mean'] - safe_stats(g_ort_right)['mean']:.2f}%" if len(g_ort_right) > 0 else "N/A"),
    ("vs Traditional Southpaw", f"{sp_stats['mean'] - safe_stats(g_southpaw)['mean']:.2f}%" if len(g_southpaw) > 0 else "N/A"),
    (None, None),
    ("KEEP TRAINING!", "*")
]

for i, (label, val) in enumerate(stats_data, start=3):
    if label is None:
        for col in [10, 11]:
            ws4.cell(i, col).fill = PatternFill("solid", fgColor=BG_ROW1)
        continue
    bg = BG_HEADER if label in ["Fighters", "Mean Win Rate", "Median Win Rate", "KEEP TRAINING!"] else (BG_ROW1 if i % 2 == 0 else BG_ROW2)
    ws4.cell(i, 10, label).fill = PatternFill("solid", fgColor=bg)
    ws4.cell(i, 11, val).fill = PatternFill("solid", fgColor=bg)
    ws4.cell(i, 11).alignment = Alignment(horizontal="center")
    ws4.row_dimensions[i].height = 18

for i, w in enumerate([5, 25, 12, 10, 7, 8, 8, 12, 3, 20, 14], 1):
    ws4.column_dimensions[get_column_letter(i)].width = w

# ============================================
# STEP 5: SAVE FILE
# ============================================
print("\nSTEP 5: Saving file...")

wb.save(OUTPUT_FILE)
print(f"Saved: {OUTPUT_FILE}")

# ============================================
# STEP 6: DISPLAY STATISTICS
# ============================================
print("\n" + "="*60)
print("FINAL STATISTICS")
print("="*60)

print(f"Total fighters: {len(df_raw)}")
print(f"Southpaw Right-Handed fighters: {len(sp_right)}")
if len(g_southpaw) > 0:
    print(f"Southpaw win rate: {g_southpaw['win_rate'].mean()*100:.2f}%")
if len(g_orthodox) > 0:
    print(f"Orthodox win rate: {g_orthodox['win_rate'].mean()*100:.2f}%")
if len(g_southpaw) > 0 and len(g_orthodox) > 0:
    print(f"Advantage: +{(g_southpaw['win_rate'].mean() - g_orthodox['win_rate'].mean())*100:.2f}%")

print("\n" + "="*60)
print("PIPELINE COMPLETE")
print("="*60)
print(f"\nFiles generated:")
print(f"   {OUTPUT_FILE} - 4 sheets")

try:
    from google.colab import files
    files.download(OUTPUT_FILE)
    print("\nFile downloaded to your computer")
except:
    print(f"\nFile saved as: {OUTPUT_FILE}")

UFC DATA CLEANING AND PROCESSING
CSV to Analysis Excel (4 sheets)

STEP 1: Loading CSV file...
Loaded 121 fighters
   Columns: ['fighter_name', 'stance', 'handedness', 'height_cm', 'reach_cm', 'weight_kg', 'wins', 'losses', 'draws', 'total_fights', 'win_rate', 'win_rate_group']

STEP 2: Cleaning data...
After cleaning: 119 fighters

STEP 3: Creating statistics groups...
Orthodox + Right: 72
Southpaw + Right (your profile): 0
Southpaw + Left: 19

STEP 4: Creating Excel file...

STEP 5: Saving file...
Saved: ufc_stance_handedness_analysis.xlsx

FINAL STATISTICS
Total fighters: 119
Southpaw Right-Handed fighters: 0
Southpaw win rate: 74.88%
Orthodox win rate: 67.07%
Advantage: +7.81%

PIPELINE COMPLETE

Files generated:
   ufc_stance_handedness_analysis.xlsx - 4 sheets


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


File downloaded to your computer
